In [ ]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [ ]:
from fast_gnn_benchmark.trainer import load_model_from_checkpoint
from fast_gnn_benchmark.data.dataset.coview_mdm import CoViewMDMDataset

import os
import torch
import boto3, json, io
import pyspark.sql.functions as F
from IPython.display import display, HTML

In [ ]:
from datetime import datetime

ckpt_dir = "/dbfs/tmp/nbraun/checkpoints/coview-mdm-128"
os.listdir(ckpt_dir)

files = [
    (os.path.getmtime(os.path.join(ckpt_dir, f)), f)
    for f in os.listdir(ckpt_dir)
]

for mtime, fname in sorted(files, reverse=True):
    print(f"{datetime.fromtimestamp(mtime):%Y-%m-%d %H:%M:%S}  {fname}")

In [ ]:
from fast_gnn_benchmark.models.link_prediction import LinkPredictionModel

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_path = os.path.join(ckpt_dir, "epoch=475-step=51884.ckpt")
raw_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

model = LinkPredictionModel.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)
model.eval()
model.to(device)

print("epoch:", raw_ckpt["epoch"])
print("global_step:", raw_ckpt["global_step"])
assert f"epoch={raw_ckpt['epoch']}-step={raw_ckpt['global_step']}" in checkpoint_path

print(model.hparams.model_parameters)

for cb_state in raw_ckpt["callbacks"].values():
    if "best_model_score" in cb_state:
        print("best_model_score (val/mrr_trigger):", cb_state["best_model_score"])

print("nb params:", sum(p.numel() for p in model.parameters()))

In [ ]:
dataset = CoViewMDMDataset()
val_split = dataset.split["valid"]

s3 = boto3.client("s3")
bucket = "mirakl-data-science-tmp2"

node2idx_raw = json.loads(s3.get_object(Bucket=bucket, Key="nbraun/datasets/coview-mdm/node2idx.json")["Body"].read())
node2idx = {int(k): v for k, v in node2idx_raw.items()}
idx2node = {v: k for k, v in node2idx.items()}

exec_mappings = json.loads(s3.get_object(Bucket=bucket, Key="nbraun/datasets/coview-mdm/exec_mappings.json")["Body"].read())

print(f"num_nodes: {dataset.num_nodes}")
print(f"node2idx entries: {len(node2idx)}")
for split in ["train", "val", "test"]:
    print(f"{split}: {len(exec_mappings[split]['code2exec'])} triggers")

In [ ]:
pos_edges = val_split["edge"]  # [N, 3] : col0=exec_code, col1=produit déclencheur, col2=produit candidat

first_trigger_id = pos_edges[0, 0].item()
trigger_node_id = pos_edges[0, 1].item()

real_execution_id = exec_mappings["val"]["code2exec"][first_trigger_id]
real_trigger_internal_id = idx2node[trigger_node_id]

print(f"trigger interne: {first_trigger_id} -> executionId prod: {real_execution_id}")
print(f"produit déclencheur (index modèle): {trigger_node_id} -> internalId réel: {real_trigger_internal_id}")

In [ ]:
trigger_category = (
    spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
    .filter(
        (F.col("internalId") == str(real_trigger_internal_id))
        & (F.col("customer_short_name") == "maisons-du-monde")
    )
    .select("t2s_best_fitting_category")
    .collect()[0]["t2s_best_fitting_category"][0]
)

print(f"catégorie du produit déclencheur: {trigger_category}")

In [ ]:
same_category_internal_ids = [
    int(row["internalId"])
    for row in (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.array_contains(F.col("t2s_best_fitting_category"), trigger_category)
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId")
        .dropDuplicates(["internalId"])
        .collect()
    )
]

all_candidates = torch.tensor([node2idx[i] for i in same_category_internal_ids if i in node2idx])
all_candidates = torch.unique(all_candidates)  # plusieurs internalId peuvent partager le même node_id
all_candidates = all_candidates[all_candidates != trigger_node_id]  # on exclut le produit déclencheur lui-même

target_edges_catalog = torch.stack([
    torch.full_like(all_candidates, trigger_node_id),
    all_candidates,
])

print(f"nombre de candidats à scorer (catégorie '{trigger_category}'): {target_edges_catalog.shape[1]}")

In [ ]:
import torch.nn.functional as TF
from torch_geometric.nn.conv.gcn_conv import gcn_norm

N = dataset.num_nodes
edge_index = dataset.data.edge_index.to(device)

norm_edge_index, norm_edge_weight = gcn_norm(edge_index, num_nodes=N, add_self_loops=True)
adj_norm = torch.sparse_coo_tensor(norm_edge_index, norm_edge_weight, (N, N)).coalesce()

def sparse_backbone_forward(backbone, x, adj_norm):
    conv_layers = backbone.conv_layers
    for layer_index, conv in enumerate(conv_layers):
        x = conv.lin(x)
        x = torch.sparse.mm(adj_norm, x)
        if conv.bias is not None:
            x = x + conv.bias
        if layer_index != len(conv_layers) - 1:
            x = TF.relu(x)
    return x

batch_size = 8192

with torch.no_grad():
    x = model.model.embedder(dataset.data.x.to(device))
    x = sparse_backbone_forward(model.model.backbone, x, adj_norm)

    logits_chunks = []
    for start in range(0, target_edges_catalog.shape[1], batch_size):
        chunk = target_edges_catalog[:, start:start + batch_size].to(device)
        logits_chunks.append(model.model.classifier(x, x, chunk))

    logits = torch.cat(logits_chunks)

print(f"logits shape: {logits.shape}")

In [ ]:
k = 12

topk_logits, topk_positions = torch.topk(logits, k)
topk_node_ids = all_candidates[topk_positions.cpu()]

print(f"Top-{k} candidats pour le trigger {first_trigger_id} (executionId prod: {real_execution_id})")
print(f"Produit déclencheur: internalId={real_trigger_internal_id}\n")

for rank, node_id in enumerate(topk_node_ids.tolist(), start=1):
    print(f"{rank}. internalId={idx2node[node_id]} (index={node_id})")

In [ ]:
topk_internal_ids = [int(idx2node[node_id]) for node_id in topk_node_ids.tolist()]
print(topk_internal_ids)

In [ ]:
from pyspark.sql import Row

def get_customer_db_name(customer_shortname: str) -> str:
    df_customer = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select(F.col("databaseName").alias("db_name"))
    )
    return df_customer.collect()[0]["db_name"]


def display_product_model(internal_ids: list[int], db_name: str, image_width: int = 150) -> None:
    ranks_df = spark.createDataFrame([Row(internalId=i, topk_rank=rank) for rank, i in enumerate(internal_ids, start=1)])

    df_products = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
        .filter(F.col("db_name") == db_name)
        .select(F.col("internalId").cast("bigint").alias("internalId"), "name", "imageUrl")
        .dropDuplicates(["internalId"])
    )

    df_products_info = (
        ranks_df
        .join(df_products, on="internalId", how="left")
        .select("topk_rank", "internalId", "name", "imageUrl")
        .orderBy("topk_rank")
    )

    for row in df_products_info.collect():
        if row["imageUrl"] is None:
            print(f"{row['topk_rank']}. internalId={row['internalId']} — pas d'image trouvée")
            continue
        display(HTML(f"<p><b>{row['topk_rank']}.</b> {row['name']} (internalId={row['internalId']})</p>"))
        display(HTML(f'<img src="{row["imageUrl"]}" width="{image_width}">'))


db_name = get_customer_db_name("maisons-du-monde")
display_product_model(topk_internal_ids, db_name=db_name)

In [ ]:
from datetime import date, timedelta

end_date = date(2026, 5, 1)
cutoff_val = end_date - timedelta(days=15)
cutoff_train = end_date - timedelta(days=30)

def get_production_candidates(execution_id: str, log_date_start: date, log_date_end: date) -> list[int]:
    df_adlog = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.ads_adlog_fct")
        .filter(
            (F.col("executionId") == execution_id)
            & F.col("__log_date").between(log_date_start, log_date_end)
        )
        .select("sponsoredProductPlacementExecutions")
    )

    df_exploded = (
        df_adlog
        .withColumn("placement", F.explode("sponsoredProductPlacementExecutions"))
        .withColumn("top_relevant_products", F.slice(F.col("placement.relevantProducts"), 1, 12))
        .select(F.posexplode("top_relevant_products").alias("rank_candidate", "candidate"))
        .select(
            (F.col("rank_candidate") + F.lit(1)).alias("rank_candidate"),
            F.col("candidate.internalId").alias("internalId_candidate"),
        )
        .orderBy("rank_candidate")
    )

    seen = set()
    deduped_candidates = []
    for row in df_exploded.collect():
        internal_id = int(row["internalId_candidate"])
        if internal_id not in seen:
            seen.add(internal_id)
            deduped_candidates.append(internal_id)

    return deduped_candidates


def display_product_production(execution_id: str, db_name: str, image_width: int = 150) -> None:
    production_internal_ids = get_production_candidates(execution_id, cutoff_train, cutoff_val)
    display_product_model(production_internal_ids, db_name=db_name, image_width=image_width)


display_product_production(real_execution_id, db_name=db_name)

%md
We do the inference on all the triggers

In [ ]:
#pos_edges = val_split["edge"] -> [N, 3] : col0=exec_code, col1=produit déclencheur, col2=produit candidat

def extract_triggers(pos_edges: torch.Tensor) -> list[dict]:

    exec_codes = pos_edges[:,0]
    trigger_node_ids = pos_edges[:,1]
    candidates_nodes_ids = pos_edges[:,2]

    sort_idx = torch.argsort(exec_codes, stable=True)

    exec_codes_sorted = exec_codes[sort_idx]
    trigger_node_ids_sorted = trigger_node_ids[sort_idx]
    candidates_node_ids_sorted = candidates_nodes_ids[sort_idx]

    unique_exec_codes, counts = torch.unique_consecutive(exec_codes_sorted, return_counts=True)

    trigger_node_id_groups = torch.split(trigger_node_ids_sorted, counts.tolist())
    candidate_node_id_groups = torch.split(candidates_node_ids_sorted, counts.tolist())

    triggers = []

    for exec_code, trigger_group, candidate_group in zip(unique_exec_codes.tolist(), trigger_node_id_groups, candidate_node_id_groups):
        assert (trigger_group == trigger_group[0]).all(), f"exec_code {exec_code} a plusieurs trigger_node_ids"
        trigger_node_id = trigger_group[0].item()
        positive_candidate_node_ids = candidate_group.tolist()

        triggers.append({
            "exec_code": exec_code,
            "trigger_node_id": trigger_node_id,
            "positive_candidate_node_ids": positive_candidate_node_ids,
        })

    return triggers

triggers = extract_triggers(pos_edges)
print(len(triggers))
print(triggers[0])

In [ ]:
def extract_categories(triggers: list[dict]) -> list[dict]:

    for trigger in triggers:
        trigger["trigger_internal_id"] = idx2node[trigger["trigger_node_id"]]

    unique_internal_ids = {t["trigger_internal_id"] for t in triggers}

    df_categories = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("internalId").isin([str(i) for i in unique_internal_ids])
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId", "t2s_best_fitting_category")
        .dropDuplicates(["internalId"])
        .collect()
    )

    category_by_internal_id = {
        int(row["internalId"]): row["t2s_best_fitting_category"][0]
        for row in df_categories
        if row["t2s_best_fitting_category"]
    }

    for trigger in triggers:
        trigger["category"] = category_by_internal_id.get(trigger["trigger_internal_id"])

    return triggers

triggers = extract_categories(triggers)
missing = sum(1 for t in triggers if t["category"] is None)
print(f"triggers sans catégorie: {missing}/{len(triggers)}")
print(triggers[0])


In [ ]:
def build_candidates_by_category(triggers: list[dict]) -> dict:

    unique_categories = {t["category"] for t in triggers if t["category"] is not None}

    df_category_products = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("t2s_best_fitting_category")[0].isin(list(unique_categories))
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    internal_ids_by_category = {c: [] for c in unique_categories}
    for row in df_category_products:
        internal_ids_by_category[row["category"]].append(int(row["internalId"]))

    candidate_node_ids_by_category = {}
    for category, internal_ids in internal_ids_by_category.items():
        node_ids = torch.tensor([node2idx[i] for i in internal_ids if i in node2idx])
        candidate_node_ids_by_category[category] = torch.unique(node_ids)  # plusieurs internalId peuvent partager le même node_id

    return candidate_node_ids_by_category

candidates_by_category = build_candidates_by_category(triggers)
pool_sizes = [c.numel() for c in candidates_by_category.values()]
print(f"nombre de catégories: {len(candidates_by_category)}")
print(f"taille moyenne du pool de candidats par catégorie: {sum(pool_sizes) / len(pool_sizes):.0f}")


In [ ]:
def run_inference(triggers: list[dict], candidates_by_category: dict) -> list[dict]:
    
    skipped_no_candidates = 0

    for trigger in triggers:
        if trigger["category"] is None:
            skipped_no_candidates += 1
            continue

        candidates_to_score = candidates_by_category[trigger["category"]]
        candidates_to_score = candidates_to_score[candidates_to_score != trigger["trigger_node_id"]]

        if candidates_to_score.numel() == 0:
            skipped_no_candidates += 1
            continue